<a href="https://colab.research.google.com/github/mejian1/ExopherGeneExpressionProfiling/blob/main/perm2genelociCREanalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install requests pandas numpy biopython pyjaspar

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 12.2 MB/s eta 0:00:00


In [2]:
import warnings
import requests
import pandas as pd
import numpy as np
from Bio.Seq import Seq
from Bio.Align import PairwiseAligner, substitution_matrices
from pyjaspar import jaspardb

warnings.filterwarnings("ignore")

HOST = "parasite.wormbase.org"
SCHEMA = "parasite_mart"
SPECIES = "caelegprjna13758"
UA = {"User-Agent": "Mozilla/5.0"}   # WormBase's Cloudflare front-end blocks default UAs

PERM_GENES = {
    "WBGene00011350": "perm-1",
    "WBGene00016636": "perm-2",
    "WBGene00016638": "perm-4",
    "WBGene00016955": "perm-5",
    "WBGene00022776": "perm-3",
}


def biomart_query(xml, formatter="TSV"):
    resp = requests.post(f"https://{HOST}/biomart/martservice",
                          data={"query": xml}, headers=UA, timeout=60)
    resp.raise_for_status()
    if resp.text.startswith("Query ERROR"):
        raise RuntimeError(resp.text)
    return resp.text


# =============================================================================
# SECTION A: shared bidirectional promoter + motif scan
# =============================================================================
def get_gene_coords(wbgene_ids):
    xml = f"""<?xml version="1.0" encoding="UTF-8"?><!DOCTYPE Query>
<Query virtualSchemaName="{SCHEMA}" formatter="TSV" header="1" uniqueRows="1" count="" datasetConfigVersion="0.6">
<Dataset name="wbps_gene" interface="default">
<Filter name="link_wbps_gene_id" value="{','.join(wbgene_ids)}"/>
<Attribute name="wbps_gene_id"/><Attribute name="external_gene_id"/>
<Attribute name="chromosome_name"/><Attribute name="start_position"/>
<Attribute name="end_position"/><Attribute name="strand"/>
</Dataset></Query>"""
    from io import StringIO
    return pd.read_csv(StringIO(biomart_query(xml)), sep="\t")


def fetch_upstream_flank(wbgene_id, bp):
    xml = f"""<?xml version="1.0" encoding="UTF-8"?><!DOCTYPE Query>
<Query virtualSchemaName="{SCHEMA}" formatter="FASTA" header="1" uniqueRows="1" count="" datasetConfigVersion="0.6">
<Dataset name="wbps_gene" interface="default">
<Filter name="link_wbps_gene_id" value="{wbgene_id}"/>
<Filter name="upstream_flank" value="{bp}"/>
<Attribute name="wbps_gene_id"/><Attribute name="gene_flank"/>
</Dataset></Query>"""
    fasta = biomart_query(xml)
    return "".join(l.strip() for l in fasta.splitlines() if not l.startswith(">"))


def get_shared_promoter(gene_a_id, gene_b_id, coords):
    """
    Returns the intergenic (bidirectional-promoter) sequence for two adjacent,
    divergently-oriented genes, in + strand orientation.
    NOTE: pulls a slightly oversized upstream flank for the + strand gene
    (gene_a here must be + strand) then trims to the validated overlap --
    see chat for why this is necessary (BioMart's flank boundary has a ~1bp
    edge convention that direct equality checks don't survive, though the
    substance of the interval is correct).
    """
    a = coords[coords.wbgene_id == gene_a_id].iloc[0]
    b = coords[coords.wbgene_id == gene_b_id].iloc[0]
    assert a.chromosome == b.chromosome, "genes on different chromosomes"
    gap = abs(min(a.start_position, a.end_position) - max(b.start_position, b.end_position))
    gap = min(abs(a.start_position - b.end_position), abs(b.start_position - a.end_position))
    padded_bp = gap + 900   # deliberately oversized; we trim to the validated overlap
    flank_plus_strand_gene = a.wbgene_id if a.strand == 1 else b.wbgene_id
    seq = fetch_upstream_flank(flank_plus_strand_gene, padded_bp)
    return seq[-gap:]        # last `gap` bases = the segment immediately adjacent
                              # to the + strand gene's start = the intergenic core


def motif_scan(promoter_seq, tax_group="nematodes", rel_score_cutoff=0.85):
    jdb = jaspardb(release="JASPAR2024")
    motifs = jdb.fetch_motifs(collection="CORE", tax_group=[tax_group])
    seq = Seq(promoter_seq)
    seq_rc = seq.reverse_complement()
    hits = []
    for m in motifs:
        m.pseudocounts = 0.8
        pssm = m.pssm
        if pssm.max <= 0:
            continue
        threshold = rel_score_cutoff * pssm.max
        for strand, s in [("+", seq), ("-", seq_rc)]:
            for pos, score in pssm.search(s, threshold=threshold, both=False):
                hits.append({"tf": m.name, "matrix_id": m.matrix_id, "strand": strand,
                             "position": pos, "rel_score": round(score / pssm.max, 3)})
    df = pd.DataFrame(hits)
    if df.empty:
        return df
    return (df.groupby(["tf", "matrix_id"])
              .agg(n_hits=("position", "count"), best_rel_score=("rel_score", "max"))
              .reset_index().sort_values("best_rel_score", ascending=False))


# =============================================================================
# SECTION B: protein-level co-evolution check across the PERM family
# =============================================================================
def get_canonical_peptides(wbgene_ids):
    xml = f"""<?xml version="1.0" encoding="UTF-8"?><!DOCTYPE Query>
<Query virtualSchemaName="{SCHEMA}" formatter="FASTA" header="1" uniqueRows="1" count="" datasetConfigVersion="0.6">
<Dataset name="wbps_gene" interface="default">
<Filter name="link_wbps_gene_id" value="{','.join(wbgene_ids)}"/>
<Attribute name="wbps_gene_id"/><Attribute name="external_gene_id"/>
<Attribute name="wbps_transcript_id"/><Attribute name="peptide"/>
</Dataset></Query>"""
    fasta = biomart_query(xml)
    recs, header, seq = {}, None, []
    for line in fasta.splitlines():
        if line.startswith(">"):
            if header:
                recs[header] = "".join(seq)
            header, seq = line[1:], []
        else:
            seq.append(line.strip())
    if header:
        recs[header] = "".join(seq)
    by_gene = {}
    for h, s in recs.items():
        gene = h.split("|")[1]
        if gene not in by_gene or len(s) > len(by_gene[gene]):
            by_gene[gene] = s
    return by_gene


def pairwise_identity_matrix(seqs_by_gene):
    aligner = PairwiseAligner()
    aligner.substitution_matrix = substitution_matrices.load("BLOSUM62")
    aligner.open_gap_score, aligner.extend_gap_score = -11, -1
    aligner.mode = "global"
    genes = list(seqs_by_gene.keys())
    rows = []
    for i, g1 in enumerate(genes):
        for g2 in genes[i + 1:]:
            aln = aligner.align(seqs_by_gene[g1], seqs_by_gene[g2])[0]
            matches = sum(1 for x, y in zip(aln[0], aln[1]) if x == y and x != "-")
            rows.append({"gene_1": g1, "gene_2": g2, "pct_identity": round(100 * matches / len(aln[0]), 1),
                         "blosum62_score": round(aln.score, 0), "aln_len": len(aln[0])})
    return pd.DataFrame(rows).sort_values("pct_identity", ascending=False)


# =============================================================================
# SECTION C: genome-wide permutation test for non-random adjacency
# =============================================================================
def get_all_protein_coding_genes():
    xml = f"""<?xml version="1.0" encoding="UTF-8"?><!DOCTYPE Query>
<Query virtualSchemaName="{SCHEMA}" formatter="TSV" header="1" uniqueRows="1" count="" datasetConfigVersion="0.6">
<Dataset name="wbps_gene" interface="default">
<Filter name="species_id_1010" value="{SPECIES}"/>
<Attribute name="wbps_gene_id"/><Attribute name="external_gene_id"/>
<Attribute name="chromosome_name"/><Attribute name="start_position"/>
<Attribute name="end_position"/><Attribute name="strand"/><Attribute name="gene_biotype"/>
</Dataset></Query>"""
    from io import StringIO
    df = pd.read_csv(StringIO(biomart_query(xml)), sep="\t")
    df.columns = ["wbgene_id", "symbol", "chrom", "start", "end", "strand", "biotype"]
    df["start"] = pd.to_numeric(df["start"], errors="coerce")   # CRITICAL: force numeric —
    df["end"] = pd.to_numeric(df["end"], errors="coerce")       # a header/dtype mismatch here
    return df.dropna(subset=["chrom", "start", "end"])          # silently breaks sort order.


def adjacency_permutation_test(gene_ids, all_genes_df, n_perm=200_000, seed=42):
    df = all_genes_df[all_genes_df.biotype == "protein_coding"].copy()
    df = df.sort_values(["chrom", "start"]).reset_index(drop=True)
    df["order_idx"] = df.groupby("chrom").cumcount()
    pos = {r.wbgene_id: (r.chrom, r.order_idx) for r in df.itertuples()}

    def count_adjacent(ids):
        ps = [pos[g] for g in ids if g in pos]
        n = 0
        for i in range(len(ps)):
            for j in range(i + 1, len(ps)):
                if ps[i][0] == ps[j][0] and abs(ps[i][1] - ps[j][1]) == 1:
                    n += 1
        return n

    observed = count_adjacent(gene_ids)
    rng = np.random.default_rng(seed)
    chrom_arr = df["chrom"].values
    order_arr = df["order_idx"].values
    n_genes, k = len(df), len(gene_ids)
    null = np.empty(n_perm, dtype=int)
    for i in range(n_perm):
        idxs = rng.choice(n_genes, size=k, replace=False)
        cs, os_ = chrom_arr[idxs], order_arr[idxs]
        n = sum(1 for a in range(k) for b in range(a + 1, k)
                if cs[a] == cs[b] and abs(int(os_[a]) - int(os_[b])) == 1)
        null[i] = n
    p_value = float(np.mean(null >= max(observed, 1)))
    return observed, null, p_value


# =============================================================================
# MAIN
# =============================================================================
ids = list(PERM_GENES.keys())

print("=" * 70, "\nSECTION A: shared bidirectional promoter + motif scan\n", "=" * 70)
coords = get_gene_coords(ids)
coords.columns = ["wbgene_id", "symbol", "chromosome", "start_position", "end_position", "strand"]
print(coords.to_string(index=False))
promoter = get_shared_promoter("WBGene00016636", "WBGene00016638", coords)
print(f"\nShared promoter length: {len(promoter)} bp")
with open("perm2_perm4_shared_promoter.fasta", "w") as f:
    f.write(">perm4_perm2_shared_bidirectional_promoter_plusStrand\n" + promoter + "\n")
motif_hits = motif_scan(promoter)
print("\nTop motif hits (nematode JASPAR2024 CORE, >=85% max PWM score):")
print(motif_hits.head(15).to_string(index=False))
motif_hits.to_csv("perm2_perm4_motif_hits.tsv", sep="\t", index=False)

print("\n" + "=" * 70, "\nSECTION B: PERM family protein co-evolution\n", "=" * 70)
peptides = get_canonical_peptides(ids)
identity_df = pairwise_identity_matrix(peptides)
print(identity_df.to_string(index=False))
identity_df.to_csv("perm_family_pairwise_identity.tsv", sep="\t", index=False)

print("\n" + "=" * 70, "\nSECTION C: non-random genomic adjacency permutation test\n", "=" * 70)
all_genes = get_all_protein_coding_genes()
observed, null, p_value = adjacency_permutation_test(ids, all_genes)
print(f"Observed adjacent pairs among the 5 PERM genes: {observed}")
print(f"Empirical p-value (permutation, N={len(null):,}): {p_value:.6f}")

print("\nDone. Files written: perm2_perm4_shared_promoter.fasta, "
      "perm2_perm4_motif_hits.tsv, perm_family_pairwise_identity.tsv")

SECTION A: shared bidirectional promoter + motif scan
     wbgene_id symbol chromosome  start_position  end_position  strand
WBGene00011350 perm-1         II         7880812       7882767       1
WBGene00016636 perm-2         IV         1126790       1128865       1
WBGene00016638 perm-4         IV         1121494       1125107      -1
WBGene00016955 perm-5         IV         5695769       5703468      -1
WBGene00022776 perm-3         IV         5393843       5394804      -1

Shared promoter length: 1683 bp

Top motif hits (nematode JASPAR2024 CORE, >=85% max PWM score):
    tf matrix_id  n_hits  best_rel_score
ceh-23  MA2129.1       2             1.0
dmd-10  MA2130.1       1             1.0
ceh-48  MA0921.2      16             1.0
ceh-43  MA2161.1       1             1.0
 elt-6  MA1439.2       2             1.0
 end-3  MA2134.1       3             1.0
 fkh-9  MA1440.1       2             1.0
 dsc-1  MA0919.2       2             1.0
 lim-7  MA1441.2       2             1.0
lin-14  MA02

In [6]:
import requests
from collections import defaultdict

def get_orthologs(gene_id):
    """Fetches orthologs for a given gene ID using the WormBase ParaSite REST API."""
    # Reverted to the working ext/rest endpoint
    url = f"https://parasite.wormbase.org/api/ext/rest/homology/id/{gene_id}?content-type=application/json"
    res = requests.get(url, headers=UA)
    if not res.ok:
        print(f"API request failed for {gene_id}: {res.status_code}")
        return []
    data = res.json()
    if not data or 'data' not in data or not data['data'][0].get('homologies'):
        print(f"No homologies returned for {gene_id}.")
        return []
    return data['data'][0]['homologies']

print("Fetching orthologs for perm-2 (WBGene00016636) and perm-4 (WBGene00016638)...")
p2_homologies = get_orthologs("WBGene00016636")
p4_homologies = get_orthologs("WBGene00016638")

# Include any ortholog (1-to-1, 1-to-many, many-to-many) and group by species
p2_map = defaultdict(list)
for h in p2_homologies:
    if 'ortholog' in h['type']:
        p2_map[h['target']['species']].append(h['target']['id'])

p4_map = defaultdict(list)
for h in p4_homologies:
    if 'ortholog' in h['type']:
        p4_map[h['target']['species']].append(h['target']['id'])

print(f"perm-2 has orthologs in {len(p2_map)} species.")
print(f"perm-4 has orthologs in {len(p4_map)} species.")

# Find common species
common_species = set(p2_map.keys()).intersection(set(p4_map.keys()))
print(f"Species in common: {len(common_species)}")
if len(common_species) > 0:
    print("Sample of common species:", list(common_species)[:5])

c_species = [sp for sp in common_species if 'caenorhabditis' in sp and sp != SPECIES]
print(f"\nFound {len(c_species)} other Caenorhabditis species with orthologs for both genes.")
print("Checking for conserved genomic adjacency and shared promoter motifs...\n")

# Test the first two closely related species (e.g., C. briggsae, C. remanei)
for sp in c_species[:2]:
    print("=" * 70)
    print(f"Species: {sp}")
    found_adjacent = False

    # Check all combinations of perm-2 and perm-4 orthologs in this species
    for g2_ortho in p2_map[sp]:
        for g4_ortho in p4_map[sp]:
            coords_sp = get_gene_coords([g2_ortho, g4_ortho])
            if len(coords_sp) < 2:
                continue

            coords_sp.columns = ["wbgene_id", "symbol", "chromosome", "start_position", "end_position", "strand"]

            if coords_sp.iloc[0].chromosome == coords_sp.iloc[1].chromosome:
                try:
                    promoter_sp = get_shared_promoter(g2_ortho, g4_ortho, coords_sp)
                    print(f"\nFound adjacent pair! perm-2_ortho: {g2_ortho}, perm-4_ortho: {g4_ortho}")
                    print(f"Shared promoter length: {len(promoter_sp)} bp")

                    motifs_sp = motif_scan(promoter_sp)
                    print("\nTop motif hits (nematode JASPAR2024 CORE, >=85% max PWM score):")
                    if not motifs_sp.empty:
                        print(motifs_sp.head(15).to_string(index=False))
                    else:
                        print("No motifs passed the cutoff.")

                    found_adjacent = True
                    break  # Stop searching this species if we found a valid adjacent pair
                except Exception:
                    # Genes are on the same chromosome but not adjacent enough
                    pass
        if found_adjacent:
            break

    if not found_adjacent:
        print("\nNo adjacent pairs found for the orthologs in this species.")


Fetching orthologs for perm-2 (WBGene00016636) and perm-4 (WBGene00016638)...
API request failed for WBGene00016636: 500
API request failed for WBGene00016638: 500
perm-2 has orthologs in 0 species.
perm-4 has orthologs in 0 species.
Species in common: 0

Found 0 other Caenorhabditis species with orthologs for both genes.
Checking for conserved genomic adjacency and shared promoter motifs...



In [ ]:
from Bio.Blast import NCBIWWW, NCBIXML
import pandas as pd
from IPython.display import display

def run_blastp(gene_name, sequence, entrez_query="txid6231[ORGN]"):
    """
    Runs BLASTp against the 'nr' database restricted to a specific Entrez query
    (default is Nematoda: txid6231).
    """
    print(f"Running BLASTp for {gene_name} (Length: {len(sequence)} aa). This may take a minute...")

    # qblast(program, database, sequence, ...)
    result_handle = NCBIWWW.qblast("blastp", "nr", sequence,
                                   entrez_query=entrez_query,
                                   hitlist_size=15)

    blast_record = NCBIXML.read(result_handle)

    hits = []
    for alignment in blast_record.alignments:
        for hsp in alignment.hsps:
            hits.append({
                "Gene": gene_name,
                "Species / Description": alignment.hit_def.split(' [')[0][:70], # Truncate long descriptions
                "E_value": hsp.expect,
                "Identity (%)": round(100 * hsp.identities / hsp.align_length, 1),
                "Align_Len": hsp.align_length
            })
            break # Only take the top High-Scoring Segment Pair (HSP) per hit

    return pd.DataFrame(hits)

# Clean up the peptide sequences (remove the trailing stop codon '*')
seq_p2 = peptides["perm-2"].replace("*", "")
seq_p4 = peptides["perm-4"].replace("*", "")

# Run BLAST for perm-2
df_p2_blast = run_blastp("perm-2", seq_p2)
print("\n--- Top BLAST hits for perm-2 ---")
display(df_p2_blast)

# Run BLAST for perm-4
df_p4_blast = run_blastp("perm-4", seq_p4)
print("\n--- Top BLAST hits for perm-4 ---")
display(df_p4_blast)


Running BLASTp for perm-2 (Length: 203 aa). This may take a minute...


In [ ]:
if 'df_p2_blast' in globals() and 'df_p4_blast' in globals():
    # Find shared species/descriptions using set intersection
    p2_hits = set(df_p2_blast['Species / Description'])
    p4_hits = set(df_p4_blast['Species / Description'])

    shared_hits = p2_hits.intersection(p4_hits)

    print(f"Found {len(shared_hits)} shared species/descriptions:")
    for hit in shared_hits:
        print(f" - {hit}")

    # Merge the dataframes to compare E-values and Identity side-by-side
    if shared_hits:
        print("\nDetailed comparison of shared hits:")
        shared_df = pd.merge(df_p2_blast, df_p4_blast,
                             on="Species / Description",
                             suffixes=('_perm2', '_perm4'))
        display(shared_df)
else:
    print("BLAST results not found. Please ensure the previous cell finishes executing.")